In [2]:
import pandas as pd
import utils
from config import Config
from preprocessing import cleaning
from analysis import plot_coordinates

In [3]:
Config.PROJECT_ROOT

'/home/gshjis/Python_projects/apartment_rent_data'

In [4]:
train = utils.load_train_data()
test = utils.load_test_data()

/home/gshjis/Python_projects/apartment_rent_data/paсkages/utils/utils/data_loader.py:16: DtypeWarning: Columns (0: address) have mixed types. Specify dtype option on import or set low_memory=False.
  train_data = pd.read_csv(train_path, encoding="windows-1252", sep=";")


Колонка curency и pricetype неинформативна

In [5]:
train.drop(columns=["currency", "price_type", "pets_allowed"], inplace=True)

In [6]:
a = set(train["cityname"].unique()) & set(train["cityname"].unique())
b = set(train["cityname"].unique())
print("Cityname intersection power ", len(a))
print("Cityname intersection train power ", len(b))


Cityname intersection power  2980
Cityname intersection train power  2980


Все города из тестовой выборки присутствуют в тренировочной

## 📊 Анализ редких категорий

Среди категориальных признаков `['category', 'fee', 'cityname', 'source']` наблюдаются редкие категории. Для дальнейшего анализа предлагается **5%** из каждой категории объединить в категорию **`other`**.

In [7]:
train = cleaning(train, ['category', 'fee', 'source'])

category: 6 редких категорий заменено на 'other'
fee: 1 редких категорий заменено на 'other'
source: 23 редких категорий заменено на 'other'


In [8]:
# utils.get_numeric_stats(train, ["amenities", "title", "id", "address", "body", "bathrooms"],n_bins=30)  

## 📈 Анализ распределений числовых признаков

### 🎯 **Ключевые наблюдения:**

#### 1. **Цена аренды (`price`)**
- **Среднее значение**: \$1527.06 (±904.25)
- **95% ДИ**: [\$1521.44, \$1532.68]
- **Характер распределения**: выраженное **экспоненциальное**
- **Интерпретация**: Большинство объектов сосредоточены в диапазоне \$600-\$2000, с длинным "хвостом" элитного жилья до \$5000+

#### 2. **Площадь (`square_feet`)**
- **Среднее значение**: 956.43 (±417.57) кв. футов
- **95% ДИ**: [953.84, 959.03]
- **Характер распределения**: **экспоненциальное**, менее выраженное
- **Интерпретация**: Преобладают компактные квартиры (500-1000 кв. футов)

#### 3. **Количество спален (`bedrooms`)**
- **Среднее значение**: 1.73 (±0.75)
- **95% ДИ**: [1.72, 1.73]
- **Характер распределения**: **скошенное**, близкое к экспоненциальному
- **Интерпретация**: Доминируют 1-2 спальни, студии и 3+ встречаются реже

### 📌 **Выводы**
- Экспоненциальный характер распределений указывает на **неравномерность рынка**: массовое бюджетное жилье + редкие элитные объекты
- Узкие доверительные интервалы (благодаря выборке 10K) обеспечивают **высокую точность оценок**
- Для подтверждения типа распределений могут быть применены **статистические тесты**

In [9]:
# utils.get_categorical_stats(train, ["amenities", "title", "id", "address", "body", "bathrooms"])  

## 📊 Анализ редких категорий

Среди категориальных признаков `['category', 'fee', 'cityname', 'source']` наблюдаются редкие категории. Для дальнейшего анализа предлагается **5%** из каждой категории объединить в категорию **`other`**.

Можно заметить что в есть 4 временных точки в которые измерений значительно больше

Объединим города и их штат

In [10]:
train['city_state'] = train['cityname'].str.cat(train['state'], sep='|')

Гипотеза: Данные по каждому городам внутри штата однородны.
Требуется проверка. 

In [11]:
from analysis import kruskal_state_city_homogeneity
kruskal_state_city_homogeneity(train, city_col="city_state")

,state,city_count,points_count,p_value,significant,alpha,effect_size_epsilon2
0,CA,222,10311,0.000000e+00,True,0.05,0.511593
7,MA,139,5030,0.000000e+00,True,0.05,0.543450
25,MD,74,5280,0.000000e+00,True,0.05,0.423230
23,NJ,138,4445,0.000000e+00,True,0.05,0.660862
12,TX,103,11257,0.000000e+00,True,0.05,0.187320
1,VA,61,8284,0.000000e+00,True,0.05,0.577614
16,FL,97,5774,3.340668e-295,True,0.05,0.293882
6,GA,59,4752,2.906232e-284,True,0.05,0.321627
2,NC,79,6300,5.861077e-271,True,0.05,0.239261
26,OH,85,4905,2.206260e-233,True,0.05,0.279674


Гипотеза не подтвердилась.

Требуется выявить какие города внтури штата различаются значимо и сколько в них объектов. На основании этого будет принято решение о разделении данных и обучении нескольких моделей и проведении A\B тестирования

In [ ]:
from analysis import dunn_posthoc_for_heterogeneous_states
res = dunn_posthoc_for_heterogeneous_states(
    train, 
    city_col="cityname",
    show_plots=False,
    min_points_city=50)
res['candidate_share_by_state']

{'CA': 0.0,
 'MA': 0.0,
 'MD': 0.0,
 'NJ': 0.0,
 'TX': 0.0,
 'VA': 0.0,
 'FL': 0.0,
 'GA': 0.0,
 'NC': 0.0,
 'OH': 0.0,
 'CO': 0.0,
 'WA': 0.0,
 'AZ': 0.0,
 'KS': 0.0,
 'LA': 0.0,
 'PA': 0.0,
 'NY': 0.0,
 'NV': 0.0,
 'TN': 0.0,
 'MO': 0.0,
 'NH': 0.0,
 'CT': 0.0,
 'SC': 0.0,
 'KY': 0.0,
 'IL': 0.0,
 'MI': 0.0,
 'OK': 0.0,
 'UT': 0.0,
 'WI': 0.0,
 'AR': 0.0,
 'ND': 0.0,
 'AL': 0.0,
 'IN': 0.0,
 'VT': 0.0,
 'MN': 0.0,
 'OR': 0.0,
 'ID': 0.0,
 'MS': 0.0,
 'NM': 0.0,
 'IA': 0.0,
 'MT': 0.0,
 'ME': 0.0}

Исследуем текстовые описание внутри одного штата с помощью LDA

In [22]:
for city in res['significant_cities_by_state']['CA']:
    n_points = len(train[train['cityname'] == city])
    city_median = train[train['cityname'] == city]['price'].median()
    state_median = train[train['state'] == 'CA']['price'].median()
    deviation = (city_median - state_median) / state_median
    print(f"{city}: {n_points} точек, отклонение = {deviation:.2%}")

San Francisco: 133 точек, отклонение = 59.45%
Culver City: 101 точек, отклонение = 79.50%
West Hollywood: 78 точек, отклонение = 59.34%
Santa Monica: 161 точек, отклонение = 64.01%
Beverly Hills: 58 точек, отклонение = 64.40%
Redwood City: 51 точек, отклонение = 75.63%
